<h1><font color="#113D68" size=6>Deep Learning en Python con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 4. Mejorar la Generalización en modelos</font></h1>

<h1><font color="#113D68" size=4>7. Pytorch con Scikit-Learn
</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. Resumen de la librería skorch](#section1)
* [2. Evaluar modelos de Deep Learning con validación cruzada](#section2)
* [3. k-Fold con scikit-learn](#section3)
* [4. Grid Search](#section4)
* [5. Pipeline](#section5)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

PyTorch es una biblioteca popular para el deep learning en Python, pero el enfoque de la biblioteca es el deep learning, no todo el aprendizaje automático. De hecho, busca el minimalismo, centrándose solo en lo que necesitas para definir y construir modelos de deep learning de manera rápida y simple. La biblioteca Scikit-learn en Python está construida sobre SciPy para cálculos numéricos eficientes. Es una biblioteca completamente equipada para el aprendizaje automático de propósito general y proporciona muchas utilidades útiles en el desarrollo de modelos de deep learning. No menos importantes son:

- Evaluación de modelos utilizando métodos de re-muestreo como la validación cruzada k-fold
- Búsqueda y evaluación eficientes de hiperparámetros del modelo
- Conectar múltiples pasos de un flujo de trabajo de aprendizaje automático en una Pipeline

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch tutorials.](https://pytorch.org/tutorials/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Documentation: [`torch.utils.data`](https://pytorch.org/docs/stable/data.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Tutorial: [Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Tutorial: [_Writing Custom Datasets, DataLoaders and Transforms_](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Resumen de la librería skorch</font>

Este método proporciona una forma poderosa de aprovechar las fortalezas tanto de PyTorch como de scikit-learn en proyectos de aprendizaje automático.

- Módulo `skorch`:
  - Construido para integrar PyTorch con scikit-learn

  - Permite usar modelos PyTorch como modelos scikit-learn

- Ventajas de `skorch`:
  - Facilita el uso de modelos PyTorch en flujos de trabajo de scikit-learn
  
  - Simplifica la integración de diferentes herramientas de aprendizaje automático

- Ejemplo de uso: `NeuralNetClassifier`
  - Envoltorio para redes neuronales de clasificación en PyTorch

  - Permite usar modelos PyTorch en scikit-learn

- Requisitos de instalación:
  - PyTorch

  - skorch

  - scikit-learn

- Beneficios de este enfoque:
  - Combina la flexibilidad de PyTorch con las utilidades de scikit-learn

  - Facilita el uso de técnicas de validación cruzada y selección de modelos de scikit-learn con modelos PyTorch

In [1]:
!pip install -q skorch  # Colab no lo trae instalado

Si estás usando `conda`, `skorch` solo está disponible en el canal `forge`:

```bash
    conda install -c conda-forge pytorch skorch scikit-learn
```

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`skorch`](https://skorch.readthedocs.io/en/stable/)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. Evaluar modelos de Deep Learning con validación cruzada</font>

Este enfoque facilita la integración de modelos PyTorch en flujos de trabajo de scikit-learn, combinando la flexibilidad de PyTorch con la facilidad de uso de scikit-learn.

- Clases envoltorio de skorch:
  - `NeuralNet`: Clase base

  - `NeuralNetClassifier`: Para clasificación general

  - `NeuralNetBinaryClassifier`: Para clasificación binaria

  - `NeuralNetRegressor`: Para regresión

- Funcionalidad de las clases envoltorio:
  - Toman un modelo PyTorch como argumento

  - Permiten especificar función de pérdida y optimizador

  - Proporcionan bucle de entrenamiento automático

- Función de pérdida utilizada:
  - `torch.nn.BCEWithLogitsLoss`

  - Predeterminada en `NeuralNetBinaryClassifier`

  - Combina función sigmoide con entropía cruzada binaria

- Proceso de entrenamiento simplificado:
  - Llamar a `fit()` con características de entrada y objetivo
  
  - El envoltorio inicializa y entrena el modelo automáticamente

In [2]:
# Descarga de datos (necesario en Colab): se guardan en ./Datasets
import os, urllib.request

os.makedirs('Datasets', exist_ok=True)
urls = {
    'sonar.csv':
        'https://raw.githubusercontent.com/jbrownlee/Datasets/master/sonar.csv',
}
for nombre, url in urls.items():
    if not os.path.exists(f'Datasets/{nombre}'):
        urllib.request.urlretrieve(url, f'Datasets/{nombre}')

In [3]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from skorch import NeuralNetBinaryClassifier

data = pd.read_csv("Datasets/sonar.csv", header=None)
X = data.iloc[:, 0:60]
y = data.iloc[:, 60]

encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)

X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

class SonarClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(60, 60)
        self.act1 = nn.ReLU()
        self.layer2 = nn.Linear(60, 60)
        self.act2 = nn.ReLU()
        self.layer3 = nn.Linear(60, 60)
        self.act3 = nn.ReLU()
        self.output = nn.Linear(60, 1)

    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.act2(self.layer2(x))
        x = self.act3(self.layer3(x))
        x = self.output(x)
        return x

# NeuralNetBinaryClassifier usa BCEWithLogitsLoss: por eso el modelo devuelve logits (sin Sigmoid)
model = NeuralNetBinaryClassifier(
    SonarClassifier,
    max_epochs=150,
    lr=0.1,
    verbose=False
)

model.fit(X, y)

,module,<class '__mai...arClassifier'>
,criterion,<class 'torch...thLogitsLoss'>
,train_split,<skorch.datas...x76768096d590>
,threshold,0.5
,optimizer,<class 'torch.optim.sgd.SGD'>
,lr,0.1
,max_epochs,150
,batch_size,128
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. k-Fold con scikit-learn</font>

Este enfoque combina la flexibilidad de PyTorch con la facilidad de uso de scikit-learn para evaluación de modelos.

- Evaluación del diseño del modelo con validación cruzada k-fold:
  - Divide el conjunto de datos en k porciones

  - Usa una porción como conjunto de test, el resto para entrenamiento

  - Entrena un modelo desde cero en cada iteración

  - Obtiene una puntuación de evaluación

- Características clave:
  - Usa `NeuralNetBinaryClassifier` de skorch

  - Parámetro `verbose=False` para evitar exceso de información

  - Utiliza `cross_val_score` de scikit-learn

  - Imprime la puntuación media de validación

- Beneficios:
  - Simplifica el proceso de validación cruzada

  - Integra seamlessly modelos PyTorch con herramientas de scikit-learn
  
  - Reduce la cantidad de código necesario para implementar k-fold

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)

In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

model = NeuralNetBinaryClassifier(
    SonarClassifier,
    max_epochs=150,
    lr=0.1,
    verbose=False
)

kfold = StratifiedKFold(n_splits=5, shuffle=True)
results = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')
print(results)

[0.5952381  0.61904762 0.66666667 0.6097561  0.65853659]


Veamos la media y la desviación estándar de las puntuaciones de la validación cruzada:

In [5]:
print("mean = %.3f; std = %.3f" % (results.mean(), results.std()))

mean = 0.630; std = 0.028


Un buen modelo debería producir una puntuación alta (en este caso, un accuracy cercana a 1) y una baja desviación estándar. Una alta desviación estándar significa que el modelo no es muy consistente con diferentes conjuntos de prueba.

En comparación, la siguiente es una implementación equivalente con un modelo de red neuronal en `scikit-learn`:

In [6]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(hidden_layer_sizes=(60, 60, 60), activation='relu', max_iter=150, learning_rate_init=0.001)

kfold = StratifiedKFold(n_splits=5, shuffle=True)
results = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')
print("mean = %.3f; std = %.3f" % (results.mean(), results.std()))

/home/santi/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/santi/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/santi/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(


mean = 0.851; std = 0.055


/home/santi/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/santi/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section4"></a>
# <font color="#004D7F" size=6>4. Grid Search</font>

Este enfoque proporciona una manera poderosa de optimizar modelos PyTorch utilizando herramientas familiares de scikit-learn.

- Personalización avanzada de modelos PyTorch con skorch y scikit-learn:
  - La función del modelo en `NeuralNetBinaryClassifier` puede aceptar múltiples argumentos

  - Permite mayor flexibilidad en la construcción del modelo

- Uso de búsqueda en cuadrícula:
  - Evalúa diferentes configuraciones del modelo

  - Reporta la combinación con mejor rendimiento estimado

- Modificación del modelo PyTorch:
  - Acepta un parámetro para determinar la profundidad de la red

- Características clave:

  - Uso de `GridSearchCV` para búsqueda de hiperparámetros

  - Evaluación de diferentes configuraciones de red y tasas de aprendizaje

- Consideraciones:
  - El tiempo de ejecución puede ser significativo debido a múltiples entrenamientos
  
  - Importante balancear la exhaustividad de la búsqueda con el tiempo de computación

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

In [7]:
class SonarClassifier(nn.Module):
    def __init__(self, n_layers=3):
        super().__init__()
        self.layers = []
        self.acts = []
        for i in range(n_layers):
            self.layers.append(nn.Linear(60, 60))
            self.acts.append(nn.ReLU())
            self.add_module(f"layer{i}", self.layers[-1])  # registrar para que aparezcan en model.parameters()
            self.add_module(f"act{i}", self.acts[-1])
        self.output = nn.Linear(60, 1)

    def forward(self, x):
        for layer, act in zip(self.layers, self.acts):
            x = act(layer(x))
        x = self.output(x)
        return x

- Problema con componentes de PyTorch:
  - No visibles en `model.parameters()`

  - Puede afectar el entrenamiento

- Soluciones:
  1. Usar `self.add_module()` para registrar componentes

  2. Utilizar `nn.ModuleList()` en lugar de listas Python

- Parámetros variados en el ejemplo:
  1. `n_layers`: Controla la profundidad de la red (en el modelo PyTorch)

  2. `lr`: Tasa de aprendizaje (en el envoltorio skorch)

  3. `max_epochs`: Número de épocas de entrenamiento (en el envoltorio)

- Sintaxis para pasar parámetros:
  - Doble guion bajo (`__`) para parámetros del modelo PyTorch

  - Ejemplo: `module__n_layers`

  - Parámetros del optimizador (ej. `optimizer__weight_decay`)

In [8]:
from sklearn.model_selection import GridSearchCV

model = NeuralNetBinaryClassifier(
    SonarClassifier,
    max_epochs=150,
    lr=0.1,
    verbose=False
)

param_grid = {
    'module__n_layers': [1, 3, 5],  # module__<arg> -> argumento del constructor del modelo
    'lr': [0.1, 0.01, 0.001, 0.0001],
    'max_epochs': [100, 150],
}

grid_search = GridSearchCV(model, param_grid, scoring='accuracy', verbose=1, cv=3, n_jobs=-1)
result = grid_search.fit(X, y)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


- Consideraciones sobre la ejecución de búsqueda en cuadrícula:
  - Puede llevar un tiempo considerable

  - Prueba todas las combinaciones de parámetros

  - Utiliza validación cruzada de 3 pliegues para cada combinación

- Utilidad:
  - Valiosa para el diseño de modelos

  - No recomendada para ejecuciones frecuentes

- Resultados mostrados tras la finalización:
  1. Rendimiento del mejor modelo

  2. Combinación de configuraciones del mejor modelo

  3. Rendimiento de todas las combinaciones de parámetros

- Interpretación de resultados:
  - `best_score_`: Mejor rendimiento logrado

  - `best_params_`: Configuración óptima encontrada

  - Tabla de resultados: Muestra todas las combinaciones y sus puntuaciones

In [9]:
print("Mejor: %f usando %s" % (result.best_score_, result.best_params_))
means = result.cv_results_['mean_test_score']
stds = result.cv_results_['std_test_score']
params = result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) con: %r" % (mean, stdev, param))

Mejor: 0.581712 usando {'lr': 0.0001, 'max_epochs': 100, 'module__n_layers': 1}
0.500552 (0.081418) con: {'lr': 0.1, 'max_epochs': 100, 'module__n_layers': 1}
0.533678 (0.003611) con: {'lr': 0.1, 'max_epochs': 100, 'module__n_layers': 3}
0.533678 (0.003611) con: {'lr': 0.1, 'max_epochs': 100, 'module__n_layers': 5}
0.543685 (0.061480) con: {'lr': 0.1, 'max_epochs': 150, 'module__n_layers': 1}
0.534023 (0.061891) con: {'lr': 0.1, 'max_epochs': 150, 'module__n_layers': 3}
0.533678 (0.003611) con: {'lr': 0.1, 'max_epochs': 150, 'module__n_layers': 5}
0.519462 (0.034476) con: {'lr': 0.01, 'max_epochs': 100, 'module__n_layers': 1}
0.533678 (0.003611) con: {'lr': 0.01, 'max_epochs': 100, 'module__n_layers': 3}
0.514631 (0.030549) con: {'lr': 0.01, 'max_epochs': 100, 'module__n_layers': 5}
0.519462 (0.034476) con: {'lr': 0.01, 'max_epochs': 150, 'module__n_layers': 1}
0.533678 (0.003611) con: {'lr': 0.01, 'max_epochs': 150, 'module__n_layers': 3}
0.533678 (0.003611) con: {'lr': 0.01, 'max_epo

Puedes ver que la búsqueda en cuadrícula descubrió que usar una tasa de aprendizaje de 0.001 con 150 épocas y solo una capa oculta logró la mejor puntuación de validación cruzada de aproximadamente 65% en este problema.

<a id="section5"></a>
# <font color="#004D7F" size=6>5. Pipeline</font>


Dado que el Pipeline permite usar el modelo de PyTorch con scikit-learn, también puedes usar la estandarización de scikit-learn en tiempo real y crear un pipeline de aprendizaje automático:

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('float32', FunctionTransformer(func=lambda X: torch.tensor(X, dtype=torch.float32), validate=False)),
    ('sonarmodel', model.initialize()),
])

- Nuevo objeto `pipe`:
  - Modelo de scikit-learn con funcionalidad extendida

  - Aplica una estandarización antes de pasar datos a la red neuronal

  - Permite ejecutar búsqueda en cuadrícula con ajustes en la especificación de parámetros

- Consideraciones sobre tipos de datos:
  - Arreglos NumPy generalmente son flotantes de 64 bits

  - Escalador de scikit-learn devuelve arreglos NumPy

  - Necesidad de conversión de tipo en medio del pipeline

- Uso de `FunctionTransformer`:
  - Realiza la conversión de tipo necesaria

  - Se integra en el pipeline para mantener la consistencia de tipos


In [11]:
param_grid = {
    'sonarmodel__module__n_layers': [1, 3, 5],
    'sonarmodel__lr': [0.1, 0.01, 0.001, 0.0001],
    'sonarmodel__max_epochs': [100, 150],
}

- Nomenclatura en pipelines de scikit-learn:
  - Cada paso se refiere por un nombre específico (ej. 'scaler', 'sonarmodel')

  - Los parámetros para la tubería deben incluir estos nombres

- Estructura de nombres de parámetros en la búsqueda en cuadrícula:

  - Formato: `[nombre_paso]__[componente]__[parámetro]`

  - Ejemplo: `sonarmodel__module__n_layers`

    - 'sonarmodel': Parte de la tubería (envoltorio skorch)

    - 'module': Componente dentro del envoltorio (modelo PyTorch)
    
    - 'n_layers': Parámetro específico del modelo

- Uso de doble guion bajo:
  - Separa los niveles jerárquicos en la nomenclatura de parámetros

In [12]:
grid_search = GridSearchCV(pipe, param_grid, scoring='accuracy', verbose=1, cv=3, n_jobs=-1)
result = grid_search.fit(X, y)
print("Mejor: %f usando %s" % (result.best_score_, result.best_params_))
means = result.cv_results_['mean_test_score']
stds = result.cv_results_['std_test_score']
params = result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) con: %r" % (mean, stdev, param))

Fitting 3 folds for each of 24 candidates, totalling 72 fits


Mejor: 0.587095 usando {'sonarmodel__lr': 0.1, 'sonarmodel__max_epochs': 100, 'sonarmodel__module__n_layers': 3}
0.572671 (0.104447) con: {'sonarmodel__lr': 0.1, 'sonarmodel__max_epochs': 100, 'sonarmodel__module__n_layers': 1}
0.587095 (0.097595) con: {'sonarmodel__lr': 0.1, 'sonarmodel__max_epochs': 100, 'sonarmodel__module__n_layers': 3}
0.533678 (0.003611) con: {'sonarmodel__lr': 0.1, 'sonarmodel__max_epochs': 100, 'sonarmodel__module__n_layers': 5}
0.544030 (0.121542) con: {'sonarmodel__lr': 0.1, 'sonarmodel__max_epochs': 150, 'sonarmodel__module__n_layers': 1}
0.568047 (0.116347) con: {'sonarmodel__lr': 0.1, 'sonarmodel__max_epochs': 150, 'sonarmodel__module__n_layers': 3}
0.553002 (0.029301) con: {'sonarmodel__lr': 0.1, 'sonarmodel__max_epochs': 150, 'sonarmodel__module__n_layers': 5}
0.586611 (0.095273) con: {'sonarmodel__lr': 0.01, 'sonarmodel__max_epochs': 100, 'sonarmodel__module__n_layers': 1}
0.509938 (0.050646) con: {'sonarmodel__lr': 0.01, 'sonarmodel__max_epochs': 100, 

<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>